# API examples

`smt-optim` proposes three different APIs, each with its own trade-off between ease of use and customizability. In order of increasing complexity, the three APIs are:
- Function API
- Component-based API
- Ask-Tell API

In [1]:
from smt_optim.benchmarks.registry import get_problem

problem = get_problem("Branin1")

## Functional API
The functional API enables users to initiate an optimization process through a single method call. The following example starts a constrained optimization problem. The `minimize` method returns a `State` object containing the optimization dataset (final DoE). Although the minimize method is simple, it offers less flexibility than the other APIs.

In [3]:
from smt_optim import minimize

constraint = [
    {
        "fun": [problem.constraints[0][-1]],
        "upper": 0.0        # the following can also be defined: "lower" or "equal"
    },
]

state = minimize(
    [problem.objective[-1]],
    design_space=problem.bounds,
    constraints=constraint,
    max_budget=20,
    method="mfsego"
)

sample = state.get_best_sample(ctol=1e-4)
print(f"best sample = \n{sample}")

          iter         budget           fmin           rscv       fidelity        gp_time       acq_time
             1              4    3.02823e+01      0.000e+00              1          0.172          1.923
             2              5    3.00077e+01      0.000e+00              1          0.294          0.116
             3              6    2.53419e+01      0.000e+00              1          0.344          0.133
             4              7    1.38640e+01      0.000e+00              1          0.345          0.253
             5              8    1.38640e+01      0.000e+00              1          0.356          0.419
             6              9    1.36488e+01      0.000e+00              1          0.338          0.583
             7             10    1.36341e+01      1.433e-07              1          0.328          0.427
             8             11    1.36341e+01      1.993e-08              1          0.369          0.419
             9             12    1.36341e+01      1.993

## Component-Based API
The component-based API enables users to define the surrogate model and acquisition strategy. It also makes the driver class available to the user. Note that the `minimize` method automates the driver and problem initialization, as shown below.

In [4]:
from smt_optim.core import Driver, ObjectiveConfig, ConstraintConfig, DriverConfig, Problem
from smt_optim.surrogate_models import SmtAutoModel
from smt_optim.acquisition_strategies import MFSEGO

obj_config = ObjectiveConfig(
    [problem.objective[-1]],
    type="minimize",
    surrogate=SmtAutoModel,
)

# configure the constraint
cstr_config = ConstraintConfig(
    [problem.constraints[0][-1]],
    upper=0.0,                      # g(x) <= 0
    surrogate=SmtAutoModel,         # set which GP to model this constraint
)

prob_definition = Problem(
    obj_configs=[obj_config],
    design_space=problem.bounds,    # problem bounds
    cstr_configs=[cstr_config],     # list the constraints
)

opt_config = DriverConfig(
    max_iter = 100,
    max_budget = 15,
    nt_init = 3,
    verbose = True,
    scaling = True,
)

driver = Driver(prob_definition, opt_config, MFSEGO)

state = driver.optimize()

sample = state.get_best_sample(ctol=1e-4)
print(f"best sample = \n{sample}")

          iter         budget           fmin           rscv       fidelity        gp_time       acq_time
             1              4    5.62485e+01      0.000e+00              1          0.449          0.219
             2              5    5.62485e+01      0.000e+00              1          0.680          0.444
             3              6    5.62485e+01      0.000e+00              1          0.599          0.465
             4              7    5.62485e+01      0.000e+00              1          0.575          0.390
             5              8    5.62485e+01      0.000e+00              1          0.606          0.322
             6              9    5.62485e+01      0.000e+00              1          0.530          0.243
             7             10    1.36333e+01      2.155e-05              1          0.569          0.291
             8             11    1.36333e+01      2.155e-05              1          0.507          0.579
             9             12    1.36333e+01      2.155

## Ask-Tell API
The Ask-Tell API uses the same driver and problem initialization as previously shown. Each iteration is called individually with the `iteration` class method. In the example below, the last iteration is defined entirely manually, similarly to within the driver's `iteration` class method. It provides for the maximum control and customization over the optimization process.

In [6]:
import time

from smt_optim.core import Driver, ObjectiveConfig, ConstraintConfig, DriverConfig, Problem
from smt_optim.surrogate_models import SmtAutoModel
from smt_optim.acquisition_strategies import MFSEGO


obj_config = ObjectiveConfig(
    [problem.objective[-1]],
    type="minimize",
    surrogate=SmtAutoModel,
)

# configure the constraint
cstr_config = ConstraintConfig(
    [problem.constraints[0][-1]],
    upper=0.0,                      # g(x) <= 0
    surrogate=SmtAutoModel,         # set which GP to model this constraint
)

prob_definition = Problem(
    obj_configs=[obj_config],
    design_space=problem.bounds,    # problem bounds
    cstr_configs=[cstr_config],     # list the constraints
)

opt_config = DriverConfig(
    max_iter = 100,
    max_budget = 20,
    nt_init = 3,
    verbose = True,
    scaling = True,
)

driver = Driver(prob_definition, opt_config, MFSEGO)

# ------- initializes state object -------
# generate initial DoE
driver.start_optim()
print(f"Number of samples: {len(driver.state.dataset.samples)}")


# ------- individually called iterations -------
for idx in range(3):
    driver.iteration(driver.state)

print(f"Number of samples: {len(driver.state.dataset.samples)}")


# ------- start of manually defined iteration -------
state = driver.state

# increment iteration counter
state.iter += 1

# scale data
state.scale_dataset(opt_config.scaling)

# build models
state.build_models()

# get infill
t0 = time.perf_counter()
infill = driver.strategy.get_infill(state)
t1 = time.perf_counter()
state.iter_log["acq_opt_time"] = t1 - t0

for i in range(len(infill)):
    if infill[i] is not None:
        infill[i] *= (
            driver.problem.design_space[:, 1] - driver.problem.design_space[:, 0]
        )
        infill[i] += driver.problem.design_space[:, 0]
        state.iter_log["fidelity"] = i + 1

# evaluate infill points
driver.evaluator.sample_func(infill, state)

# log iteration data
driver.call_loggers(state)
# ------- end of manually defined iteration -------

print(f"Number of samples: {len(driver.state.dataset.samples)}")

sample = state.get_best_sample(ctol=1e-4)
print(f"best sample = \n{sample}")

Number of samples: 3
          iter         budget           fmin           rscv       fidelity        gp_time       acq_time
             1              4    5.01070e+01      0.000e+00              1          0.440          0.153
             2              5    5.01070e+01      0.000e+00              1          0.584          0.402
             3              6    2.31258e+01      0.000e+00              1          0.563          0.222
Number of samples: 6
             4              7    2.31258e+01      0.000e+00              1          0.564          0.250
Number of samples: 7
best sample = 
======= sample data =======
x =             [0.43254541 0.47378314]
obj =           [23.1258357]
cstr =          [-0.00493272]
eval_time =     [8.00599992e-06 1.45800004e-06]
------- meta data -------
iter =     3
budget =     6
fidelity =     0
rscv =     0.0

